In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
subjects = pd.read_csv('data/subjects.csv')
physiological = pd.read_csv('data/timeseries.csv')
print(subjects.head())
print(physiological.head())

  SubjectID  STAI_T  STAI_S Gender Handedness  WearsGlasses  CalibrationError  \
0      S001      37      36      F          R             0          0.396519   
1      S002      45      38      F          L             0          0.937323   
2      S003      40      39      M          R             1          0.270619   
3      S004      34      30      M          L             1          0.387328   
4      S005      25      40      M          R             0          0.263703   

  BloodType  
0         B  
1         O  
2         B  
3         O  
4         O  
  SubjectID  DeviceTimestamp  CycleID     Phase  PupilDiameter     GazeX  \
0      S001                0        1  baseline       3.556082 -0.008417   
1      S001            16667        1  baseline       3.587611 -0.006079   
2      S001            33334        1  baseline       3.587292 -0.007796   
3      S001            50001        1  baseline       3.607923 -0.008068   
4      S001            66668        1  baseline  

### Subjects and physiological

In [3]:
# Number of unique subjects
print("subjects number:", subjects.SubjectID.nunique())
print("physiological number:", physiological.SubjectID.nunique())
print("physiological length:", len(physiological))

# Variables types
print("variables types:\n", subjects.dtypes)
print("physiological variables types:\n", physiological.dtypes)

# Missing values
print("missing values:\n", subjects.isnull().sum())
print("physiological missing values:\n", physiological.isnull().sum())

# Duplicate values
print("duplicate values:\n", subjects.duplicated().sum())
print("physiological duplicate values:\n", physiological.duplicated().sum())

# Duplicates rate on physiological data
print("physiological duplicates rate:", physiological.duplicated().sum() / len(physiological))

subjects number: 500
physiological number: 500
physiological length: 14040000
variables types:
 SubjectID               str
STAI_T                int64
STAI_S                int64
Gender                  str
Handedness              str
WearsGlasses          int64
CalibrationError    float64
BloodType               str
dtype: object
physiological variables types:
 SubjectID              str
DeviceTimestamp      int64
CycleID              int64
Phase                  str
PupilDiameter      float64
GazeX              float64
GazeY              float64
GazeZ              float64
PulseBPM           float64
PPG_SQI            float64
MotionMag          float64
dtype: object
missing values:
 SubjectID           0
STAI_T              0
STAI_S              0
Gender              0
Handedness          0
WearsGlasses        0
CalibrationError    0
BloodType           0
dtype: int64
physiological missing values:
 SubjectID          0
DeviceTimestamp    0
CycleID            0
Phase              0
Pu

#### Delete duplicates

In [4]:
# Delete duplicates
physiological = physiological.drop_duplicates()
print("physiological length after dropping duplicates:", len(physiological))

physiological length after dropping duplicates: 12000000


#### Data distribution

In [5]:
# Summary statistics
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print("summary statistics:\n", subjects.describe())
print("physiological summary statistics:\n", physiological.describe().round(4) )

summary statistics:
         STAI_T   STAI_S  WearsGlasses  CalibrationError
count 500.0000 500.0000      500.0000          500.0000
mean   40.3440  39.0260        0.2600            0.3234
std    13.4278  10.9828        0.4391            0.2410
min    20.0000  20.0000        0.0000            0.0503
25%    30.0000  31.0000        0.0000            0.1567
50%    37.0000  37.0000        0.0000            0.2624
75%    50.0000  45.2500        1.0000            0.3678
max    80.0000  77.0000        1.0000            0.9929
physiological summary statistics:
        DeviceTimestamp       CycleID  PupilDiameter         GazeX  \
count    12000000.0000 12000000.0000  12000000.0000 12000000.0000   
mean    199995666.5000        5.5000         2.8451       -0.1450   
std     115472367.9501        2.8723         1.7923        0.4170   
min             0.0000        1.0000        -1.0000       -1.0000   
25%      99997833.2500        3.0000         2.6829       -0.2439   
50%     199995666.5000    

In [ ]:
# Discard outliers in physiological data
# lower_bound = physiological['HeartRate'].quantile(0.1)
# upper_bound = physiological['HeartRate'].quantile(0.9)
# physiological = physiological[(physiological['HeartRate'] >= lower_bound) & (physiological['HeartRate'] <= upper_bound)]
# print("physiological length after discarding outliers:", len(physiological))

In [6]:
# Discart low quality pulse signals
physiological = physiological[physiological.PPG_SQI > 0.5]
print("physiological length after discarding low quality pulse signals:", len(physiological))

# Discart too high motion magnitude
physiological = physiological[physiological.MotionMag <= 0.7]
print("physiological length after discarding low quality motion signals:", len(physiological))

# Transform GazeZ in degrees
physiological['GazeZAngle'] = physiological['GazeZ'].apply(lambda z: np.degrees(np.arccos(z)))

# Discard off-Gaze samples
physiological = physiological[physiological.GazeZAngle <= 30]
print("physiological length after discarding off-Gaze samples:", len(physiological))

# Correct pupils diameters based on gaze
physiological['PupilDiameterCorrection'] = physiological.apply(lambda row: row['PupilDiameter']/row['GazeZ'] if row['GazeZ'] > 0 else None, axis=1)



physiological length after discarding low quality pulse signals: 11560802
physiological length after discarding low quality motion signals: 11555322
physiological length after discarding off-Gaze samples: 8724032


In [8]:
# Delete outliers
lower_bound = physiological['PupilDiameterCorrection'].quantile(0.1)
upper_bound = physiological['PupilDiameterCorrection'].quantile(0.9)
physiological = physiological[(physiological['PupilDiameterCorrection'] >= lower_bound) & (physiological['PupilDiameterCorrection'] <= upper_bound)]
print("physiological length after deleting outliers:", len(physiological))

physiological length after deleting outliers: 4489553


In [ ]:
# Remove implausible values in physiological data
physiological = physiological[(physiological.PupilDiameterCorrection >= 2) & (physiological.PupilDiameterCorrection <= 9)]
physiological = physiological[(physiological.PulseBPM >= 40) & (physiological.PulseBPM <= 300)]
print("physiological length after removing implausible values:", len(physiological))

physiological length after removing implausible values: 4489553


##### Maybe I would apply some filtering on the data, using the best practice for the specific case. I will skip it as I would need to further investigate the best practices in this specific case.